### Install dependencies

In [1]:
! pip install --upgrade --quiet accelerate bitsandbytes transformers

## Set model variant and configuration

In [1]:
from transformers import BitsAndBytesConfig
import torch

model_variant = "4b-it"  # @param ["4b-it", "27b-text-it"]
model_id = f"google/medgemma-{model_variant}"

use_quantization = True  # @param {type: "boolean"}

# @markdown Set `is_thinking` to `True` to turn on thinking mode. **Note:** Thinking is supported for the 27B variant only.
is_thinking = False  # @param {type: "boolean"}

# If running the 27B variant in Google Colab, check if the runtime satisfies
# memory requirements
if "27b" in model_variant and google_colab:
    if not ("A100" in torch.cuda.get_device_name(0) and use_quantization):
        raise ValueError(
            "Runtime has insufficient memory to run the 27B variant. "
            "Please select an A100 GPU and use 4-bit quantization."
        )

model_kwargs = dict(
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

if use_quantization:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

## Run inference on images and text

This section demonstrates running inference on image-based tasks using multimodal variants.

**Note:** Proceed to [Run inference on text only](#scrollTo=tcyXG4lTpY4X) if you have selected the 27B text-only variant.

In [ ]:
if "text" in model_variant:
    raise ValueError(
        "You are using a text-only variant which does not support multimodal "
        "inputs. Please proceed to the 'Run inference on text only' section."
    )

**Specify image and text inputs**

In [ ]:
import os
from PIL import Image
from IPython.display import Image as IPImage, display, Markdown

prompt = "Describe this polyp image, where the polyp actually is is described in the mask(second picture), output in chinese"  # @param {type: "string"}
# Image attribution: Stillwaterising, CC0, via Wikimedia Commons
image_url = "https://upload.wikimedia.org/wikipedia/commons/c/c8/Chest_Xray_PA_3-8-2010.png"  # @param {type: "string"}
! wget -nc -q {image_url}
image_filename = os.path.basename(image_url)
image = Image.open(image_filename)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


**Format conversation**

In [10]:
system_instruction = "You are an expert polyp segmenter and analyst."

messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": system_instruction}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": prompt},
            {"type": "image", "image": Image.open("./1.png")},
            {"type": "image", "image": Image.open("./1_mask.png")}
        ]
    }
]

The following standalone examples demonstate how to use the model both directly and with the [`pipeline`](https://huggingface.co/docs/transformers/en/main_classes/pipelines) API. The `pipeline` API provides a simple way to use the model for inference while abstracting away complex details,  while directly using the model gives you complete control over the inference process, including preprocessing and postprocessing. In practice, you should select the method that is best suited for your use case.

**Run model with the `pipeline` API**

In [5]:
from transformers import pipeline

# Update quantization config for CPU compatibility
if use_quantization and "quantization_config" in model_kwargs:
    model_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4"
    )

pipe = pipeline(
    "image-text-to-text",
    model=model_id,
    model_kwargs=model_kwargs,
)

pipe.model.generation_config.do_sample = False

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


In [12]:

output = pipe(text=messages, max_new_tokens=300)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
# display(IPImage(filename=image_filename, height=300))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


---

**[ User ]**

息肉是什么

---

**[ MedGemma ]**

Based on the image, there appears to be a polyp present in the colon.

A polyp is a growth of tissue that can form on the lining of the colon or rectum. They are often benign (non-cancerous), but some polyps can develop into cancer over time.

In the image, the polyp appears as a raised, irregular area on the colon lining. Further investigation, such as a colonoscopy with biopsy, would be needed to determine the type of polyp and whether it is cancerous.


---

**Run the model directly**

In [12]:
from transformers import AutoModelForImageTextToText, AutoProcessor

model = AutoModelForImageTextToText.from_pretrained(
    model_id,
    **model_kwargs,
)
processor = AutoProcessor.from_pretrained(model_id)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.64G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

In [21]:
# Prepare inputs for the model
inputs = processor(
    text=prompt,
    images=image,
    return_tensors="pt"
).to(model.device, dtype=torch.bfloat16)

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=300, do_sample=False)

# Decode the generated output
response = processor.decode(output[0], skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}"))
display(IPImage(filename=image_filename, height=300))
display(Markdown(f"---\n\n**[ MedGemma ]**\n\n{response}\n\n---"))

ValueError: Unable to infer channel dimension format

## Run inference on text only

This section demonstrates running inference on text-based tasks.

**Specify text prompt and format conversation**

In [ ]:
from IPython.display import Markdown

prompt = "How do you differentiate bacterial from viral pneumonia?"  # @param {type: "string"}

role_instruction = "You are a helpful medical assistant."
if "27b" in model_variant and is_thinking:
    system_instruction = f"SYSTEM INSTRUCTION: think silently if needed. {role_instruction}"
    max_new_tokens = 1500
else:
    system_instruction = role_instruction
    max_new_tokens = 500

messages = [
    {
        "role": "system",
        "content": system_instruction
    },
    {
        "role": "user",
        "content": prompt
    }
]

The following standalone examples demonstate how to use the model both directly and with the [`pipeline`](https://huggingface.co/docs/transformers/en/main_classes/pipelines) API. The `pipeline` API provides a simple way to use the model for inference while abstracting away complex details,  while directly using the model gives you complete control over the inference process, including preprocessing and postprocessing. In practice, you should select the method that is best suited for your use case.

**Run model with the `pipeline` API**

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs=model_kwargs,
)

pipe.model.generation_config.do_sample = False

In [ ]:
output = pipe(messages, max_new_tokens=max_new_tokens)
response = output[0]["generated_text"][-1]["content"]

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

How do you differentiate bacterial from viral pneumonia?

---

**[ MedGemma ]**

Okay, I can explain the key differences between bacterial and viral pneumonia. As a medical assistant, I understand that this is a crucial distinction because the treatment strategies differ significantly. While a definitive diagnosis often requires lab tests, clinical presentation and history can provide strong clues.

Here's a breakdown of how we typically differentiate between the two:

**1. Onset & Symptoms:**

*   **Bacterial Pneumonia:**
    *   **Onset:** Often **sudden and abrupt**. Patients might feel fine one day and acutely ill the next.
    *   **Symptoms:**
        *   High fever (often > 102°F / 39°C).
        *   Shaking chills.
        *   Productive cough: Coughing up thick, often colored (yellow, green, rust-colored) sputum.
        *   Sharp, stabbing pleuritic chest pain (pain that worsens with deep breaths or coughing).
        *   Rapid breathing (tachypnea) and shortness of breath.
        *   Fatigue and malaise.
        *   May experience sweating.
        *   Sometimes confusion (especially in older adults).

*   **Viral Pneumonia:**
    *   **Onset:** Usually **gradual**, often starting with cold-like symptoms that progress over several days.
    *   **Symptoms:**
        *   Lower-grade fever (though it can sometimes be high).
        *   Dry, non-productive cough (hacking cough, may produce small amounts of clear or whitish mucus).
        *   Muscle aches (myalgia) and headache.
        *   Sore throat.
        *   General feeling of being unwell (malaise).
        *   Shortness of breath, but often less severe initially than bacterial.
        *   Chest discomfort may be present but is often less sharp or localized than pleuritic pain.
        *   Wheezing can sometimes occur.

**2. Patient History & Risk Factors:**

*   **Bacterial Pneumonia:**
    *   Often follows a recent upper respiratory infection (like a cold or flu) that worsens or doesn't improve.
    *   More common in individuals with underlying chronic health conditions (COPD, heart disease, diabetes, kidney disease), weakened immune systems (due to illness, medications like steroids, or chemotherapy), smokers, and older adults.
    *   Hospital

---

**Run the model directly**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    **model_kwargs,
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

input_len = inputs["input_ids"].shape[-1]

with torch.inference_mode():
    generation = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generation = generation[0][input_len:]

response = tokenizer.decode(generation, skip_special_tokens=True)

display(Markdown(f"---\n\n**[ User ]**\n\n{prompt}\n\n---"))
if "27b" in model_variant and is_thinking:
    thought, response = response.split("<unused95>")
    thought = thought.replace("<unused94>thought\n", "")
    display(Markdown(f"**[ MedGemma thinking ]**\n\n{thought}\n\n---"))
display(Markdown(f"**[ MedGemma ]**\n\n{response}\n\n---"))

---

**[ User ]**

How do you differentiate bacterial from viral pneumonia?

---

**[ MedGemma ]**

Okay, let's break down how healthcare professionals differentiate between bacterial and viral pneumonia. It's important to remember that I'm an AI and cannot provide medical advice. This information is for educational purposes only. A definitive diagnosis always requires a healthcare professional.

Differentiating bacterial and viral pneumonia can be tricky because symptoms often overlap. However, there are key differences in presentation, history, and diagnostic tests that help clinicians distinguish between them.

Here's a breakdown of the key differentiating factors:

**1. Clinical Presentation & History:**

*   **Onset:**
    *   **Bacterial:** Often sudden and abrupt onset. Patients may feel fine one day and severely ill the next.
    *   **Viral:** Typically more gradual onset, often starting with cold-like symptoms (runny nose, cough, sore throat) that then progress to pneumonia symptoms.
*   **Symptoms:**
    *   **Bacterial:** High fever (often >102°F / 39°C), shaking chills, productive cough (producing thick, often colored sputum - yellow, green, rust-colored), shortness of breath, chest pain (often sharp, pleuritic - worse with breathing).
    *   **Viral:** Lower-grade fever (though can be high), dry cough (less likely to produce thick sputum), muscle aches (myalgia), headache, fatigue, sore throat. Shortness of breath can occur but might be less severe initially than in bacterial pneumonia.
*   **Patient History:**
    *   **Bacterial:** Less likely to have preceding upper respiratory infection (URI) symptoms. May have underlying conditions like COPD, heart failure, diabetes, or be immunocompromised.
    *   **Viral:** Often preceded by URI symptoms. More common in younger children and older adults. Can occur in outbreaks (e.g., influenza season).

**2. Physical Examination:**

*   **Lung Sounds:**
    *   **Bacterial:** Often localized findings. Crackles (rales), rhonchi, or decreased breath sounds over the affected area.
    *   **Viral:** Lung sounds might be more diffuse (spread throughout) or relatively normal, especially early on. Wheezing can sometimes be present.
*   **General Appearance:**
    *   **Bacterial:** Patients often appear more acutely ill, toxic-looking.
    *   **Viral:** Patients might appear less acutely ill initially,

---

# Next steps

Explore the other [notebooks](https://github.com/google-health/medgemma/blob/main/notebooks) to learn what else you can do with the model.